# ЛР 7

## Импорты

In [1]:
!pip install -q watermark transformers gdown datasets nlpaug

from tqdm import tqdm
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.metrics.pairwise import cosine_similarity
from transformers import BertTokenizer, BertModel, get_linear_schedule_with_warmup
import nlpaug.augmenter.word as naw
%reload_ext watermark
%watermark -v -p torch,transformers

import warnings
warnings.filterwarnings('ignore')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.5/410.5 kB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 46.9 MB/s eta 0:00:00
Python implementation: CPython
Python version       : 3.12.13
IPython version      : 7.34.0

torch       : 2.10.0+cu128
transformers: 5.0.0



## Загрузка и работа с данными

In [2]:
!gdown 1S6qMioqPJjyBLpLVz4gmRTnJHnjitnuV
!gdown 1zdmewp7ayS4js4VtrJEHzAheSW-5NBZv

df = pd.read_csv("reviews.csv")
print("Размер датасета:", df.shape)
df.head(2)

Downloading...
From: https://drive.google.com/uc?id=1S6qMioqPJjyBLpLVz4gmRTnJHnjitnuV
To: /content/apps.csv
100% 134k/134k [00:00<00:00, 78.2MB/s]
Downloading...
From: https://drive.google.com/uc?id=1zdmewp7ayS4js4VtrJEHzAheSW-5NBZv
To: /content/reviews.csv
100% 7.17M/7.17M [00:00<00:00, 120MB/s]
Размер датасета: (15746, 11)


,userName,userImage,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,sortOrder,appId
0,Andrew Thomas,https://lh3.googleusercontent.com/a-/AOh14GiHd...,Update: After getting a response from the deve...,1,21,4.17.0.3,2020-04-05 22:25:57,"According to our TOS, and the term you have ag...",2020-04-05 15:10:24,most_relevant,com.anydo
1,Craig Haines,https://lh3.googleusercontent.com/-hoe0kwSJgPQ...,Used it for a fair amount of time without any ...,1,11,4.17.0.3,2020-04-04 13:40:01,It sounds like you logged in with a different ...,2020-04-05 15:11:35,most_relevant,com.anydo


In [3]:
def to_sentiment(rating):
    rating = int(rating)
    if rating <= 2:
        return 0
    elif rating == 3:
        return 1
    else:
        return 2

df['sentiment'] = df.score.apply(to_sentiment)
class_names = ['negative', 'neutral', 'positive']

df_train, df_test = train_test_split(df, test_size=0.1, random_state=42)
df_val, df_test = train_test_split(df_test, test_size=0.5, random_state=42)

df_train = df_train[:1000]
df_val = df_val[:100]
df_test = df_test[:100]

print(f"Train: {len(df_train)}, Val: {len(df_val)}, Test: {len(df_test)}")

Train: 1000, Val: 100, Test: 100


In [4]:
PRE_TRAINED_MODEL_NAME = 'bert-base-cased'
tokenizer = BertTokenizer.from_pretrained(PRE_TRAINED_MODEL_NAME)

MAX_LEN = 160
BATCH_SIZE = 16
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device: cuda


In [5]:
class GPReviewDataset(Dataset):
    def __init__(self, reviews, targets, tokenizer, max_len):
        self.reviews = reviews
        self.targets = targets
        self.tokenizer = tokenizer
        self.max_len = max_len
    def __len__(self):
        return len(self.reviews)
    def __getitem__(self, item):
        review = str(self.reviews[item])
        target = self.targets[item]
        encoding = self.tokenizer(
            review,
            add_special_tokens=True,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_token_type_ids=False,
            return_attention_mask=True,
            return_tensors="pt",
        )
        return {
            'review_text': review,
            "input_ids": encoding["input_ids"].squeeze(0),
            'attention_mask': encoding['attention_mask'].flatten(),
            'targets': torch.tensor(target, dtype=torch.long)
        }

def create_data_loader(df, tokenizer, max_len, batch_size):
    ds = GPReviewDataset(
        reviews=df.content.to_numpy(),
        targets=df.sentiment.to_numpy(),
        tokenizer=tokenizer,
        max_len=max_len
    )
    return DataLoader(ds, batch_size=batch_size, num_workers=0)

train_loader = create_data_loader(df_train, tokenizer, MAX_LEN, BATCH_SIZE)
val_loader = create_data_loader(df_val, tokenizer, MAX_LEN, BATCH_SIZE)
test_loader = create_data_loader(df_test, tokenizer, MAX_LEN, BATCH_SIZE)

## Модель

In [6]:
class SentimentClassifier(nn.Module):
    def __init__(self, n_classes):
        super(SentimentClassifier, self).__init__()
        self.bert = BertModel.from_pretrained(PRE_TRAINED_MODEL_NAME)
        self.drop = nn.Dropout(p=0.3)
        self.out = nn.Linear(self.bert.config.hidden_size, n_classes)
    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        output = self.drop(outputs["pooler_output"])
        return self.out(output)

model = SentimentClassifier(len(class_names)).to(device)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [7]:
def train_epoch(model, loader, loss_fn, optimizer, device, scheduler, n_examples):
    model.train()
    losses = []
    correct = 0
    loop = tqdm(loader, desc="Training batch", leave=False)
    for d in loop:
        input_ids = d["input_ids"].to(device)
        attention_mask = d["attention_mask"].to(device)
        targets = d["targets"].to(device)
        outputs = model(input_ids, attention_mask)
        _, preds = torch.max(outputs, dim=1)
        loss = loss_fn(outputs, targets)
        correct += torch.sum(preds == targets)
        losses.append(loss.item())
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
        loop.set_postfix(loss=loss.item())
    return correct.double() / n_examples, np.mean(losses)

def eval_model(model, loader, loss_fn, device, n_examples):
    model.eval()
    losses = []
    correct = 0
    with torch.no_grad():
        for d in loader:
            input_ids = d["input_ids"].to(device)
            attention_mask = d["attention_mask"].to(device)
            targets = d["targets"].to(device)
            outputs = model(input_ids, attention_mask)
            _, preds = torch.max(outputs, dim=1)
            loss = loss_fn(outputs, targets)
            correct += torch.sum(preds == targets)
            losses.append(loss.item())
    return correct.double() / n_examples, np.mean(losses)

## Обучение

In [8]:
EPOCHS = 5
optimizer = AdamW(model.parameters(), lr=2e-5)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)
loss_fn = nn.CrossEntropyLoss().to(device)

In [9]:
for epoch in range(EPOCHS):
    train_acc, train_loss = train_epoch(model, train_loader, loss_fn, optimizer, device, scheduler, len(df_train))
    val_acc, val_loss = eval_model(model, val_loader, loss_fn, device, len(df_val))
    print(f"Epoch {epoch+1}: train_acc={train_acc:.4f}, val_acc={val_acc:.4f}")

torch.save(model.state_dict(), 'bert_finetuned.bin')

Epoch 1: train_acc=0.3570, val_acc=0.4700


Epoch 2: train_acc=0.5060, val_acc=0.5800


Epoch 3: train_acc=0.6310, val_acc=0.6100


Epoch 4: train_acc=0.7400, val_acc=0.5800


Epoch 5: train_acc=0.7990, val_acc=0.6400


In [10]:
test_acc, test_loss = eval_model(model, test_loader, loss_fn, device, len(df_test))
print(f"Базовая модель ({PRE_TRAINED_MODEL_NAME}) — точность на тесте: {test_acc:.4f}")

Базовая модель (bert-base-cased) — точность на тесте: 0.5800


## 1 задание

In [11]:
def get_cls_embedding(bert_model, tokenizer, word, device):
    tokens = tokenizer(word, return_tensors='pt', add_special_tokens=True).to(device)
    with torch.no_grad():
        outputs = bert_model(**tokens)
    return outputs.pooler_output.squeeze().cpu().numpy()

bert_original = BertModel.from_pretrained(PRE_TRAINED_MODEL_NAME).to(device)
bert_original.eval()

words = ['good', 'bad']
emb_orig = {w: get_cls_embedding(bert_original, tokenizer, w, device) for w in words}
emb_finetuned = {}
for w in words:
    tokens = tokenizer(w, return_tensors='pt', add_special_tokens=True).to(device)
    with torch.no_grad():
        outputs = model.bert(**tokens)
    emb_finetuned[w] = outputs.pooler_output.squeeze().cpu().numpy()

sim_orig = cosine_similarity([emb_orig['good']], [emb_orig['bad']])[0][0]
sim_finetuned = cosine_similarity([emb_finetuned['good']], [emb_finetuned['bad']])[0][0]

print(f"Сходство 'good' и 'bad' до обучения: {sim_orig:.4f}")
print(f"Сходство 'good' и 'bad' после обучения: {sim_finetuned:.4f}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Сходство 'good' и 'bad' до обучения: 0.9883
Сходство 'good' и 'bad' после обучения: 0.3433


## 2 задание

In [12]:
PRE_TRAINED_MODEL_NAME2 = 'bert-base-uncased'
tokenizer2 = BertTokenizer.from_pretrained(PRE_TRAINED_MODEL_NAME2)

train_loader2 = create_data_loader(df_train, tokenizer2, MAX_LEN, BATCH_SIZE)
val_loader2 = create_data_loader(df_val, tokenizer2, MAX_LEN, BATCH_SIZE)
test_loader2 = create_data_loader(df_test, tokenizer2, MAX_LEN, BATCH_SIZE)

class SentimentClassifier2(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        self.bert = BertModel.from_pretrained(PRE_TRAINED_MODEL_NAME2)
        self.drop = nn.Dropout(p=0.3)
        self.out = nn.Linear(self.bert.config.hidden_size, n_classes)
    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        output = self.drop(outputs["pooler_output"])
        return self.out(output)

model2 = SentimentClassifier2(len(class_names)).to(device)
optimizer2 = AdamW(model2.parameters(), lr=2e-5)
total_steps2 = len(train_loader2) * EPOCHS
scheduler2 = get_linear_schedule_with_warmup(optimizer2, num_warmup_steps=0, num_training_steps=total_steps2)

for epoch in range(EPOCHS):
    train_acc, train_loss = train_epoch(model2, train_loader2, loss_fn, optimizer2, device, scheduler2, len(df_train))
    val_acc, val_loss = eval_model(model2, val_loader2, loss_fn, device, len(df_val))
    print(f"Epoch {epoch+1}: train_acc={train_acc:.4f}, val_acc={val_acc:.4f}")

test_acc2, _ = eval_model(model2, test_loader2, loss_fn, device, len(df_test))
print(f"Модель {PRE_TRAINED_MODEL_NAME2} — точность на тесте: {test_acc2:.4f}")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

...

Epoch 1: train_acc=0.4190, val_acc=0.5800


Epoch 2: train_acc=0.6330, val_acc=0.5900


Epoch 3: train_acc=0.7570, val_acc=0.6200


Epoch 4: train_acc=0.8310, val_acc=0.7000


Epoch 5: train_acc=0.8850, val_acc=0.6800
Модель bert-base-uncased — точность на тесте: 0.6100


## 3 задание

In [13]:
import nltk
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('wordnet')
nltk.download('omw-1.4')

aug = naw.SynonymAug(aug_src='wordnet')

def augment_texts(texts, n_aug=2):
    augmented = []
    for text in texts:
        for _ in range(n_aug):
            aug_text = aug.augment(text)
            if isinstance(aug_text, list):
                aug_text = aug_text[0]
            augmented.append(aug_text)
    return augmented

minority_class = 1
minority_df = df_train[df_train['sentiment'] == minority_class]
print(f"Количество примеров класса {minority_class}: {len(minority_df)}")

aug_texts = augment_texts(minority_df['content'].tolist(), n_aug=2)
aug_rows = []
for i, text in enumerate(aug_texts):
    new_row = minority_df.iloc[i % len(minority_df)].copy()
    new_row['content'] = text
    aug_rows.append(new_row)

df_train_aug = pd.concat([df_train, pd.DataFrame(aug_rows)], ignore_index=True)
print("Размер после аугментации:", len(df_train_aug))
print("Новое распределение классов:")
print(df_train_aug['sentiment'].value_counts())

[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


Количество примеров класса 1: 299
Размер после аугментации: 1598
Новое распределение классов:
sentiment
1    897
2    384
0    317
Name: count, dtype: int64


In [14]:
train_loader_aug = create_data_loader(df_train_aug, tokenizer, MAX_LEN, BATCH_SIZE)

model_aug = SentimentClassifier(len(class_names)).to(device)
optimizer_aug = AdamW(model_aug.parameters(), lr=2e-5)
total_steps_aug = len(train_loader_aug) * EPOCHS
scheduler_aug = get_linear_schedule_with_warmup(optimizer_aug, num_warmup_steps=0, num_training_steps=total_steps_aug)

for epoch in range(EPOCHS):
    train_acc, train_loss = train_epoch(model_aug, train_loader_aug, loss_fn, optimizer_aug, device, scheduler_aug, len(df_train_aug))
    val_acc, val_loss = eval_model(model_aug, val_loader, loss_fn, device, len(df_val))
    print(f"Epoch {epoch+1}: train_acc={train_acc:.4f}, val_acc={val_acc:.4f}")

test_acc_aug, _ = eval_model(model_aug, test_loader, loss_fn, device, len(df_test))
print(f"Модель с аугментацией — точность на тесте: {test_acc_aug:.4f}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch 1: train_acc=0.6414, val_acc=0.3700


Epoch 2: train_acc=0.6677, val_acc=0.3700


Epoch 3: train_acc=0.6758, val_acc=0.3700


Epoch 4: train_acc=0.7315, val_acc=0.3600


Epoch 5: train_acc=0.6602, val_acc=0.6100
Модель с аугментацией — точность на тесте: 0.5700


In [15]:
print("Сравнение результатов")
print(f"Базовая модель ({PRE_TRAINED_MODEL_NAME}):          {test_acc:.4f}")
print(f"Модель {PRE_TRAINED_MODEL_NAME2}:                   {test_acc2:.4f}")
print(f"Модель с аугментацией:                              {test_acc_aug:.4f}")

Сравнение результатов
Базовая модель (bert-base-cased):          0.5800
Модель bert-base-uncased:                   0.6100
Модель с аугментацией:                              0.5700
